# DaTSCAN — Fase 2: preprocesamiento común por protocolos

Este notebook parte de los folds ya auditados. Convierte cada NIfTI a orientación RAS, remuestrea a 2.5 mm, normaliza de manera robusta, localiza el componente principal de la cabeza, recorta a `96×96×64` y guarda cada volumen en `float16`.

Primero se ejecuta una auditoría visual estratificada por protocolo. El procesamiento completo se habilita después de revisar esas imágenes.

## 0. Dependencias

In [ ]:
# Descomente solamente si faltan paquetes:
# %pip install numpy pandas nibabel scipy matplotlib seaborn

## 1. Librerías y rutas reales

In [ ]:
from pathlib import Path
import os
import json, time, warnings
import matplotlib.pyplot as plt
import nibabel as nib
from nibabel.processing import resample_to_output
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.ndimage import binary_closing, binary_fill_holes, gaussian_filter, label

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
DATA_ROOT = Path(r'C:\Users\DELL\OneDrive\Escritorio\kaggle\parkinson')
PROJECT_DIR = DATA_ROOT / 'latent_protocol_cv'
FOLDS_CSV = PROJECT_DIR / 'outputs' / 'train_protocol_folds.csv'
METADATA_CSV = PROJECT_DIR / 'outputs' / 'protocol_metadata_clustered.csv'
# Se usa una carpeta v2 para impedir que se reutilicen recortes de la versión anterior.
PREPROCESS_DIR = PROJECT_DIR / 'preprocessed_96x96x64_v2'
QC_DIR = PROJECT_DIR / 'preprocessing_qc_v2'
PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SPACING = (2.5, 2.5, 2.5)
OUTPUT_SHAPE = (96, 96, 64)
HEAD_MASK_MIN_THRESHOLD = 0.02
HEAD_MASK_PERCENTILE = 20
PERCENTILE_SCALE = 99.5
RANDOM_STATE = 20260910
SAMPLES_PER_CLUSTER = 2

for path in (FOLDS_CSV, METADATA_CSV):
    print(path, '| existe:', path.exists())
    if not path.exists():
        raise FileNotFoundError(path)

## 2. Cargar el manifiesto maestro y verificarlo

In [ ]:
folds = pd.read_csv(FOLDS_CSV)
metadata = pd.read_csv(METADATA_CSV)
manifest = folds.merge(metadata[['uid','nifti_path']], on='uid', how='left', validate='one_to_one')

required = {'uid','target','protocol_cluster','fold','nifti_path'}
missing_columns = required.difference(manifest.columns)
if missing_columns:
    raise KeyError(f'Faltan columnas: {sorted(missing_columns)}')
if manifest['nifti_path'].isna().any():
    raise ValueError('Hay UID sin ruta NIfTI.')
missing_files = manifest.loc[~manifest['nifti_path'].map(lambda p: Path(p).exists())]
if len(missing_files):
    display(missing_files.head())
    raise FileNotFoundError(f'No se localizaron {len(missing_files)} NIfTI.')
if manifest['uid'].duplicated().any():
    raise ValueError('El manifiesto contiene UID duplicados.')
if (manifest.groupby('protocol_cluster')['fold'].nunique() > 1).any():
    raise ValueError('Un clúster aparece dividido entre folds.')

print('Estudios:', len(manifest))
display(pd.crosstab(manifest['protocol_cluster'], manifest['target'], margins=True))
display(manifest.groupby(['fold','protocol_cluster']).size().rename('n').reset_index())

## 3. Funciones de preprocesamiento
La localización usa únicamente intensidades de la propia imagen, nunca la etiqueta. Se suaviza el volumen, se forma una máscara de baja intensidad y se conserva su mayor componente conectado. El centro de su caja delimitadora determina el recorte y queda aproximadamente en `(48,48,32)`. Esto reduce el desplazamiento producido por captación extracerebral intensa.

In [ ]:
def robust_normalize(volume, percentile=99.5):
    volume = np.nan_to_num(volume.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    volume[volume < 0] = 0
    positive = volume[volume > 0]
    if positive.size < 100:
        raise ValueError('Volumen sin suficientes vóxeles positivos.')
    scale = float(np.percentile(positive, percentile))
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError(f'Escala robusta inválida: {scale}')
    return np.clip(volume / scale, 0, 1), scale

def head_component_center(volume):
    smoothed = gaussian_filter(volume.astype(np.float32), sigma=2.0)
    positive = smoothed[smoothed > 0]
    if positive.size < 100:
        raise ValueError('No se puede construir la máscara de cabeza.')
    threshold = max(HEAD_MASK_MIN_THRESHOLD, float(np.percentile(positive, HEAD_MASK_PERCENTILE)))
    mask = smoothed >= threshold
    mask = binary_closing(mask, iterations=2)
    labeled, n_components = label(mask)
    if n_components == 0:
        raise ValueError('No se encontraron componentes en la máscara.')
    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    largest = labeled == int(np.argmax(sizes))
    largest = binary_fill_holes(largest)
    coords = np.argwhere(largest)
    bbox_min = coords.min(axis=0)
    bbox_max = coords.max(axis=0)
    center = (bbox_min + bbox_max) / 2.0
    return tuple(float(v) for v in center), tuple(int(v) for v in bbox_min), tuple(int(v) for v in bbox_max), threshold, float(largest.mean())

def crop_pad_around(volume, center, output_shape=(96,96,64)):
    output = np.zeros(output_shape, dtype=np.float32)
    center = np.rint(center).astype(int)
    starts = center - np.asarray(output_shape)//2
    src_slices, dst_slices = [], []
    for axis, (start, size, source_size) in enumerate(zip(starts, output_shape, volume.shape)):
        src_start, src_end = max(0, start), min(source_size, start+size)
        dst_start = max(0, -start)
        dst_end = dst_start + max(0, src_end-src_start)
        src_slices.append(slice(src_start, src_end))
        dst_slices.append(slice(dst_start, dst_end))
    output[tuple(dst_slices)] = volume[tuple(src_slices)]
    return output, tuple(int(v) for v in starts)

def preprocess_nifti(path):
    original = nib.load(path, mmap=True)
    original_shape = tuple(int(v) for v in original.shape[:3])
    original_spacing = tuple(float(v) for v in original.header.get_zooms()[:3])
    canonical = nib.as_closest_canonical(original)
    resampled = resample_to_output(canonical, voxel_sizes=TARGET_SPACING, order=1)
    volume = np.asarray(resampled.dataobj, dtype=np.float32)
    volume, scale = robust_normalize(volume, PERCENTILE_SCALE)
    center, bbox_min, bbox_max, mask_threshold, component_fraction = head_component_center(volume)
    cropped, crop_start = crop_pad_around(volume, center, OUTPUT_SHAPE)
    if cropped.shape != OUTPUT_SHAPE or not np.isfinite(cropped).all():
        raise ValueError('Salida inválida después del recorte.')
    info = {
        'original_shape': original_shape, 'original_spacing': original_spacing,
        'canonical_orientation': ''.join(nib.aff2axcodes(canonical.affine)),
        'resampled_shape': tuple(int(v) for v in volume.shape),
        'head_center_x':center[0], 'head_center_y':center[1], 'head_center_z':center[2],
        'head_bbox_min':bbox_min, 'head_bbox_max':bbox_max,
        'head_mask_threshold':mask_threshold, 'head_component_fraction':component_fraction,
        'crop_start_x':crop_start[0], 'crop_start_y':crop_start[1], 'crop_start_z':crop_start[2],
        'p995_scale':scale, 'output_nonzero_fraction':float(np.mean(cropped>0)),
        'output_mean':float(cropped.mean()), 'output_max':float(cropped.max())
    }
    return cropped.astype(np.float16), info

## 4. Prueba técnica de un estudio

In [ ]:
example = manifest.iloc[0]
test_volume, test_info = preprocess_nifti(example['nifti_path'])
print('UID:', example['uid'], '| protocolo:', example['protocol_cluster'], '| fold:', example['fold'])
print('Forma/dtype/rango:', test_volume.shape, test_volume.dtype, (test_volume.min(), test_volume.max()))
display(pd.Series(test_info, name='valor').to_frame())

## 5. Auditoría visual por protocolo
Se muestran dos estudios de cada clúster. Se incluyen tres alturas axiales, una proyección axial máxima y los planos coronal y sagital centrales. El cerebro debe quedar completo y centrado; la proyección permite confirmar que la captación estriatal permanece dentro del recorte.

In [ ]:
qc_sample = (manifest.groupby('protocol_cluster', group_keys=False)
             .sample(n=SAMPLES_PER_CLUSTER, random_state=RANDOM_STATE))
qc_rows = []
fig, axes = plt.subplots(len(qc_sample), 6, figsize=(19, 3*len(qc_sample)))
for plot_row, (_, row) in enumerate(qc_sample.reset_index(drop=True).iterrows()):
    volume, info = preprocess_nifti(row['nifti_path'])
    x, y, z = np.asarray(OUTPUT_SHAPE)//2
    views = [volume[:,:,z-8].T, volume[:,:,z].T, volume[:,:,z+8].T, volume.max(axis=2).T, volume[:,y,:].T, volume[x,:,:].T]
    titles = ['Axial z=24', 'Axial z=32', 'Axial z=40', 'MIP axial', 'Coronal y=48', 'Sagital x=48']
    for j, (view,title) in enumerate(zip(views,titles)):
        axes[plot_row,j].imshow(view, cmap='inferno', origin='lower', vmin=0, vmax=1)
        axes[plot_row,j].set_title(f"C{row['protocol_cluster']} | {row['uid']} | {title}", fontsize=9)
        axes[plot_row,j].axis('off')
    qc_rows.append({'uid':row['uid'],'target':row['target'],'protocol_cluster':row['protocol_cluster'],**info})
plt.tight_layout()
plt.savefig(QC_DIR/'audit_by_protocol.png', dpi=160, bbox_inches='tight')
plt.show()
qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(QC_DIR/'audit_by_protocol.csv', index=False)
display(qc_df)

## 6. Decisión manual obligatoria
Cambie a `True` solamente después de revisar el mosaico anterior. Si alguna captación aparece cortada, fuera del centro o con orientación sospechosa, deténgase y comparta la imagen.

In [ ]:
VISUAL_AUDIT_APPROVED = False
print('Auditoría aprobada:', VISUAL_AUDIT_APPROVED)

## 7. Procesamiento completo reiniciable
Cada estudio se guarda por separado. Si la ejecución se interrumpe, los archivos válidos existentes se omiten. El tamaño máximo sin compresión ronda 1.2 MB por estudio.

In [ ]:
if not VISUAL_AUDIT_APPROVED:
    raise RuntimeError('Primero revise las imágenes y cambie VISUAL_AUDIT_APPROVED = True.')

records, errors = [], []
start_time = time.time()
for i, row in manifest.iterrows():
    output_path = PREPROCESS_DIR / f"{row['uid']}.npz"
    try:
        reused = False
        if output_path.exists():
            with np.load(output_path) as saved:
                existing_volume = saved['volume'] if 'volume' in saved else None
                valid = existing_volume is not None and existing_volume.shape == OUTPUT_SHAPE and np.isfinite(existing_volume).all()
            if valid:
                reused = True
                info = {'original_shape':None,'original_spacing':None,'canonical_orientation':None,'resampled_shape':None,'head_center_x':None,'head_center_y':None,'head_center_z':None,'head_bbox_min':None,'head_bbox_max':None,'head_mask_threshold':None,'head_component_fraction':None,'crop_start_x':None,'crop_start_y':None,'crop_start_z':None,'p995_scale':None,'output_nonzero_fraction':float(np.mean(existing_volume>0)),'output_mean':float(existing_volume.mean()),'output_max':float(existing_volume.max())}
        if not reused:
            volume, info = preprocess_nifti(row['nifti_path'])
            np.savez_compressed(output_path, volume=volume)
        records.append({'uid':row['uid'],'target':int(row['target']),'protocol_cluster':int(row['protocol_cluster']),'fold':int(row['fold']),'nifti_path':row['nifti_path'],'processed_path':str(output_path.resolve()),'reused':reused,**info})
    except Exception as exc:
        errors.append({'uid':row['uid'],'nifti_path':row['nifti_path'],'error_type':type(exc).__name__,'error':str(exc)})
    if (i+1)%25 == 0 or i+1 == len(manifest):
        elapsed = (time.time()-start_time)/60
        print(f"{i+1}/{len(manifest)} | correctos={len(records)} | errores={len(errors)} | {elapsed:.1f} min")
        pd.DataFrame(records).to_csv(PROJECT_DIR/'preprocessing_manifest_checkpoint.csv', index=False)
        pd.DataFrame(errors).to_csv(PROJECT_DIR/'preprocessing_errors_checkpoint.csv', index=False)

processed_manifest = pd.DataFrame(records)
error_df = pd.DataFrame(errors)
processed_manifest.to_csv(PROJECT_DIR/'preprocessing_manifest.csv', index=False)
error_df.to_csv(PROJECT_DIR/'preprocessing_errors.csv', index=False)
print('Finalizado. Correctos:',len(processed_manifest),'| errores:',len(error_df))
display(error_df.head(20))

## 8. Auditoría numérica final

In [ ]:
if len(error_df):
    print('ATENCIÓN: existen errores; no continúe a extracción de características.')
summary = processed_manifest.groupby('protocol_cluster').agg(
    n=('uid','size'), reused=('reused','sum'),
    nonzero_median=('output_nonzero_fraction','median'),
    mean_median=('output_mean','median'), max_min=('output_max','min')
).reset_index()
display(summary)
expected_uids, actual_uids = set(manifest['uid']), set(processed_manifest['uid'])
print('Faltantes:', len(expected_uids-actual_uids))
print('Extras:', len(actual_uids-expected_uids))
print('Listos para la fase siguiente:', len(error_df)==0 and expected_uids==actual_uids)